In [2]:
pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [3]:
import faiss
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import spacy


In [4]:
nlp = spacy.load('en_core_web_sm')

In [5]:
data = open('Data3.txt').read()
data

'sample_text = """Machine Learning is a subset of Artificial Intelligence that enables systems to learn and improve from experience without being explicitly programmed.\nSupervised learning involves training models on labeled datasets, where both input features and target outputs are provided.\nCommon supervised learning algorithms include linear regression, logistic regression, decision trees, random forests, and support vector machines.\nUnsupervised learning discovers hidden patterns in unlabeled data through clustering and dimensionality reduction techniques.\nK-means clustering, hierarchical clustering, and DBSCAN are popular unsupervised learning methods.\nDeep Learning uses neural networks with multiple layers to automatically extract features from raw data.\nConvolutional Neural Networks (CNNs) are designed for image processing and computer vision tasks.\nRecurrent Neural Networks (RNNs) and Long Short-Term Memory (LSTM) networks excel at processing sequential data and time ser

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size= 100,
    chunk_overlap= 20
)

In [7]:
chunks = splitter.split_text(data)
print(chunks,type(chunks),type(chunks[0]))

['sample_text = """Machine Learning is a subset of Artificial Intelligence that enables systems to', 'enables systems to learn and improve from experience without being explicitly programmed.', 'Supervised learning involves training models on labeled datasets, where both input features and', 'input features and target outputs are provided.', 'Common supervised learning algorithms include linear regression, logistic regression, decision', 'decision trees, random forests, and support vector machines.', 'Unsupervised learning discovers hidden patterns in unlabeled data through clustering and', 'clustering and dimensionality reduction techniques.', 'K-means clustering, hierarchical clustering, and DBSCAN are popular unsupervised learning methods.', 'Deep Learning uses neural networks with multiple layers to automatically extract features from raw', 'features from raw data.', 'Convolutional Neural Networks (CNNs) are designed for image processing and computer vision tasks.', 'Recurrent Neur

In [8]:
embedding_model = SentenceTransformer(
    model_name_or_path='sentence-transformers/all-miniLM-L6-V2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
embeddings = embedding_model.encode(chunks).astype('float32')

In [10]:
embeddings

array([[-0.02838418,  0.00449589, -0.01720257, ...,  0.10232824,
         0.07816096, -0.09840076],
       [ 0.01560274,  0.0263623 , -0.02546506, ...,  0.10216852,
         0.02557372, -0.0734313 ],
       [ 0.00556227, -0.04957569, -0.03187537, ...,  0.04997924,
         0.05631769, -0.04995145],
       ...,
       [ 0.01242197,  0.08071334, -0.05262596, ..., -0.03879428,
         0.01175592, -0.03927461],
       [-0.0205646 , -0.06712151,  0.01596265, ..., -0.14173196,
        -0.06897087,  0.03440064],
       [ 0.014768  , -0.03163011, -0.00998562, ..., -0.05420229,
         0.04494205, -0.01978256]], dtype=float32)

In [11]:
dimension = embeddings.shape[1]

In [12]:
faiss.normalize_L2(embeddings)

In [13]:
index = faiss.IndexFlatIP(dimension) #dot product
# index = faiss.IndexFlatL2() #eucliean distance

In [14]:
index.add(embeddings)  # stored embeddings inside Faiss DB

## Search Method(R)

In [15]:
query = "Explain Machine Learning ?"

In [16]:
query_embeddings = embedding_model.encode(query).astype('float32')

In [17]:
query_embeddings = query_embeddings.reshape(1,-1)

In [18]:
query_embeddings.shape

(1, 384)

In [19]:
faiss.normalize_L2(query_embeddings)

In [20]:


'''
1st value - > distance
2nd value - > 
'''

index.search(query_embeddings,k=5)

(array([[0.5593344 , 0.5555082 , 0.5351466 , 0.5168163 , 0.47923982]],
       dtype=float32),
 array([[21,  0,  4,  2, 25]], dtype=int64))